In [1]:
# ======================================
# GPU CHECK & SYSTEM INFO
# ======================================
import torch
import subprocess, platform

print("=" * 50)
print("SYSTEM INFO")
print("=" * 50)
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No CUDA GPU detected. Models will run on CPU.")
print("=" * 50)

SYSTEM INFO
Python: 3.13.3
PyTorch: 2.7.0+cu118
CUDA Available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.4 GB


In [2]:
# ======================================
# LOCAL FILE LISTING
# ======================================
import os

DATA_DIR = r"D:\cyberr\wustl_iiot_2021"

for dirname, _, filenames in os.walk(DATA_DIR):
    for filename in filenames:
        filepath = os.path.join(dirname, filename)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{filepath}  ({size_mb:.1f} MB)")

D:\cyberr\wustl_iiot_2021\wustl_iiot_2021.csv  (390.8 MB)


In [3]:
# ======================================
# CELL 1 - IMPORT LIBRARIES
# ======================================

import numpy as np
import pandas as pd
import os
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, accuracy_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix)
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_selection import RFE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
import shap

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [4]:
# ======================================
# CELL 2 - DATA LOADING
# ======================================
# Load the WUSTL-IIoT-2021 dataset.
# Only drop pure identifier / timestamp columns.
# Feature selection will handle the rest scientifically.

df = pd.read_csv(r"D:\cyberr\wustl_iiot_2021\wustl_iiot_2021.csv")

print("Original Shape:", df.shape)
print("Columns:", list(df.columns))

# Drop only non-numeric identifiers / timestamps
id_cols = ['StartTime', 'LastTime', 'SrcAddr', 'DstAddr',
           'Sport', 'Dport', 'Proto', 'Dir', 'state']
df = df.drop(columns=[c for c in id_cols if c in df.columns], errors='ignore')
print(f"\nAfter dropping identifiers: {df.shape}")

# Handle any remaining NaN / Inf
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

# Encode multi-class target (Traffic)
le = LabelEncoder()
df['Traffic'] = le.fit_transform(df['Traffic'])

# Separate features and target
X_full = df.drop(['Traffic', 'Target'], axis=1, errors='ignore')
y_full = df['Traffic']

print(f"Feature matrix: {X_full.shape}")
print(f"Number of classes: {y_full.nunique()}")
print(f"Class distribution:\n{y_full.value_counts().sort_index()}")

Original Shape: (1194464, 49)
Columns: ['StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'Mean', 'Sport', 'Dport', 'SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes', 'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss', 'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt', 'DIntPkt', 'Proto', 'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max', 'sDSb', 'sTtl', 'dTtl', 'sIpId', 'dIpId', 'SAppBytes', 'DAppBytes', 'TotAppByte', 'SynAck', 'RunTime', 'sTos', 'SrcJitAct', 'DstJitAct', 'Traffic', 'Target']

After dropping identifiers: (1194464, 42)
Feature matrix: (1194464, 40)
Number of classes: 5
Class distribution:
Traffic
0        212
1        259
2      78305
3       8240
4    1107448
Name: count, dtype: int64


# ═══ FEATURE SELECTION MODULE ═══
> **New Stage** — Replaces arbitrary column dropping with a multi-strategy,
> scientifically grounded feature selection pipeline.
>
> | Stage | Method | Purpose |
> |-------|--------|---------|
> | 1 | Random Forest Importance | Preliminary importance ranking |
> | 2 | Correlation Filtering | Redundancy removal (threshold 0.90) |
> | 3 | RFE with XGBoost (GPU) | Recursive subset optimisation |
> | 4 | Red Ant Algorithm | Swarm-intelligence subset search |
> | 5 | Aggregation | Voting-based final feature set |

In [5]:
# ======================================
# FEATURE SELECTION — STAGE 1
# Tree-Based Feature Importance Ranking
# ======================================
# Train a Random Forest on the full dataset and rank features
# by their Gini importance scores.

from sklearn.ensemble import RandomForestClassifier
import time

print("Stage 1: Random Forest Feature Importance Ranking")
print("=" * 55)

t0 = time.time()

# Use a stratified subsample for speed on large datasets
SAMPLE_SIZE = min(200_000, len(X_full))
idx = np.random.choice(len(X_full), SAMPLE_SIZE, replace=False)
X_sample = X_full.iloc[idx]
y_sample = y_full.iloc[idx]

rf_selector = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    class_weight='balanced'
)
rf_selector.fit(X_sample, y_sample)

# Build importance table
importances = rf_selector.feature_importances_
rf_importance_df = pd.DataFrame({
    'Feature': X_full.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)
rf_importance_df['Rank'] = rf_importance_df.index + 1

print(f"\nCompleted in {time.time()-t0:.1f}s")
print(f"\n{'Rank':<6}{'Feature':<20}{'Importance':<12}")
print("-" * 38)
for _, row in rf_importance_df.iterrows():
    print(f"{int(row['Rank']):<6}{row['Feature']:<20}{row['Importance']:.6f}")

# Store feature-to-importance mapping for later stages
rf_importance_map = dict(zip(rf_importance_df['Feature'], rf_importance_df['Importance']))
rf_selected_features = set(rf_importance_df['Feature'].tolist())  # all features initially
print(f"\nTotal features ranked: {len(rf_importance_df)}")

Stage 1: Random Forest Feature Importance Ranking

Completed in 6.5s

Rank  Feature             Importance  
--------------------------------------
1     IdleTime            0.146801
2     sTtl                0.095358
3     DAppBytes           0.056016
4     TcpRtt              0.054974
5     TotAppByte          0.054941
6     DIntPkt             0.054439
7     SrcLoss             0.047522
8     SynAck              0.047257
9     pLoss               0.033417
10    SAppBytes           0.030259
11    DstRate             0.027381
12    DstLoad             0.020819
13    SrcBytes            0.020607
14    DstJitter           0.019985
15    SrcLoad             0.019984
16    Loss                0.019779
17    DstBytes            0.018630
18    DstPkts             0.018057
19    dTtl                0.016837
20    RunTime             0.016744
21    Rate                0.016271
22    TotBytes            0.015953
23    SrcRate             0.015693
24    Max                 0.013393
25    Sum   

In [6]:
# ======================================
# FEATURE SELECTION — STAGE 2
# Correlation-Based Redundancy Removal
# ======================================
# Remove one feature from each highly-correlated pair (|r| >= 0.90),
# keeping the one with higher RF importance.

print("Stage 2: Correlation-Based Redundancy Removal")
print("=" * 55)

CORR_THRESHOLD = 0.90

t0 = time.time()

corr_matrix = X_full.corr().abs()

# Upper triangle to avoid duplicate pairs
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop_corr = set()
corr_pairs = []

for col in upper.columns:
    for idx_name in upper.index:
        val = upper.loc[idx_name, col]
        if pd.notna(val) and val >= CORR_THRESHOLD:
            # Drop the feature with LOWER RF importance
            imp_col = rf_importance_map.get(col, 0)
            imp_idx = rf_importance_map.get(idx_name, 0)
            drop_feat = col if imp_col < imp_idx else idx_name
            keep_feat = idx_name if drop_feat == col else col
            to_drop_corr.add(drop_feat)
            corr_pairs.append((idx_name, col, val, keep_feat, drop_feat))

print(f"\nHighly correlated pairs found (|r| >= {CORR_THRESHOLD}): {len(corr_pairs)}")
print(f"\n{'Feature A':<18}{'Feature B':<18}{'Corr':<8}{'Keep':<18}{'Drop':<18}")
print("-" * 80)
for a, b, r, keep, drop in corr_pairs[:25]:
    print(f"{a:<18}{b:<18}{r:.4f}  {keep:<18}{drop:<18}")
if len(corr_pairs) > 25:
    print(f"  ... and {len(corr_pairs)-25} more pairs")

features_after_corr = [f for f in X_full.columns if f not in to_drop_corr]
print(f"\nFeatures removed: {len(to_drop_corr)}  ->  {sorted(to_drop_corr)}")
print(f"Features remaining: {len(features_after_corr)}")
print(f"Completed in {time.time()-t0:.1f}s")

Stage 2: Correlation-Based Redundancy Removal

Highly correlated pairs found (|r| >= 0.9): 21

Feature A         Feature B         Corr    Keep              Drop              
--------------------------------------------------------------------------------
SrcPkts           TotPkts           1.0000  SrcPkts           TotPkts           
SrcLoad           Load              0.9955  SrcLoad           Load              
SrcLoad           SrcRate           0.9980  SrcLoad           SrcRate           
Load              SrcRate           0.9936  SrcRate           Load              
DstLoad           DstRate           0.9841  DstRate           DstLoad           
SrcLoad           Rate              0.9938  SrcLoad           Rate              
Load              Rate              0.9979  Rate              Load              
SrcRate           Rate              0.9957  Rate              SrcRate           
Mean              Sum               0.9948  Sum               Mean              
Mean          

In [7]:
# ======================================
# FEATURE SELECTION — STAGE 3
# RFE with XGBoost (GPU Accelerated)
# ======================================
# Recursive Feature Elimination using XGBoost as the estimator.
# Uses features that survived the correlation filter.

from sklearn.feature_selection import RFE
from xgboost import XGBClassifier
import time

print("Stage 3: RFE with XGBoost (GPU)")
print("=" * 55)

t0 = time.time()

# Work on the correlation-filtered feature set
X_rfe = X_full[features_after_corr].copy()

# Subsample for RFE speed
RFE_SAMPLE = min(100_000, len(X_rfe))
idx_rfe = np.random.choice(len(X_rfe), RFE_SAMPLE, replace=False)
X_rfe_sample = X_rfe.iloc[idx_rfe]
y_rfe_sample = y_full.iloc[idx_rfe]

# XGBoost estimator with GPU
xgb_rfe = XGBClassifier(
    n_estimators=100,
    max_depth=8,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=y_full.nunique(),
    tree_method='hist',
    device='cuda',
    random_state=RANDOM_SEED,
    verbosity=0
)

# Target: select ~60% of remaining features (minimum 10)
n_target = max(10, int(len(features_after_corr) * 0.6))

rfe = RFE(
    estimator=xgb_rfe,
    n_features_to_select=n_target,
    step=2,
    verbose=0
)
rfe.fit(X_rfe_sample, y_rfe_sample)

rfe_selected = [f for f, s in zip(features_after_corr, rfe.support_) if s]
rfe_ranking = dict(zip(features_after_corr, rfe.ranking_))

print(f"\nOptimal feature count: {len(rfe_selected)}")
print(f"\nRFE Selected Features:")
print("-" * 40)
for i, f in enumerate(rfe_selected, 1):
    print(f"  {i:>3}. {f}")

print(f"\nRFE Ranking (all features):")
rfe_rank_df = pd.DataFrame({
    'Feature': features_after_corr,
    'RFE_Rank': [rfe_ranking[f] for f in features_after_corr]
}).sort_values('RFE_Rank')
print(rfe_rank_df.to_string(index=False))

rfe_selected_set = set(rfe_selected)
print(f"\nCompleted in {time.time()-t0:.1f}s")

Stage 3: RFE with XGBoost (GPU)

Optimal feature count: 16

RFE Selected Features:
----------------------------------------
    1. SrcPkts
    2. DstPkts
    3. SrcBytes
    4. TotBytes
    5. DstRate
    6. SrcLoss
    7. DstLoss
    8. pLoss
    9. SrcJitter
   10. SIntPkt
   11. DIntPkt
   12. TcpRtt
   13. IdleTime
   14. sTtl
   15. dTtl
   16. DstJitAct

RFE Ranking (all features):
   Feature  RFE_Rank
   SrcPkts         1
   DstPkts         1
  SrcBytes         1
  TotBytes         1
   DstRate         1
   SrcLoss         1
   DstLoss         1
     pLoss         1
    TcpRtt         1
 SrcJitter         1
   DIntPkt         1
   SIntPkt         1
 DstJitAct         1
      dTtl         1
      sTtl         1
  IdleTime         1
TotAppByte         2
       Dur         2
      Loss         3
 SrcJitAct         3
     sIpId         4
 DAppBytes         4
     dIpId         5
 SAppBytes         5
   RunTime         6
   SrcLoad         6
 DstJitter         7
      sTos         7


In [8]:
# ======================================
# FEATURE SELECTION — STAGE 4
# Red Ant Feature Selection Algorithm
# ======================================
# Swarm intelligence optimisation inspired by ant colony search.
# Each ant selects a random feature subset, evaluates fitness,
# and the colony converges on the best subset over iterations.
#
# Fitness = Accuracy - lambda * (n_selected / n_total)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import time

print("Stage 4: Red Ant Feature Selection Algorithm")
print("=" * 55)

t0 = time.time()

# Parameters
N_ANTS = 20                # number of ants per iteration
N_ITERATIONS = 15          # total iterations
LAMBDA_PENALTY = 0.05      # simplicity penalty
MIN_FEATURES = 5           # minimum features an ant can select
EVAPORATION = 0.3          # pheromone evaporation rate
ALPHA = 1.0                # pheromone influence
BETA = 2.0                 # heuristic (RF importance) influence

# Use features that passed correlation filter
ant_features = features_after_corr.copy()
n_features = len(ant_features)

# Subsample for speed
ANT_SAMPLE = min(50_000, len(X_full))
idx_ant = np.random.choice(len(X_full), ANT_SAMPLE, replace=False)
X_ant = X_full[ant_features].iloc[idx_ant]
y_ant = y_full.iloc[idx_ant]

# Initialize pheromone trails (uniform)
pheromone = np.ones(n_features)

# Heuristic info: RF importance (normalized)
heuristic = np.array([rf_importance_map.get(f, 0.001) for f in ant_features])
heuristic = heuristic / heuristic.sum()

# Quick classifier for fitness evaluation
def evaluate_subset(feature_indices, X_data, y_data):
    """Evaluate a feature subset using RF accuracy (3-fold CV)."""
    if len(feature_indices) == 0:
        return 0.0
    X_sub = X_data.iloc[:, feature_indices]
    clf = RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
    scores = cross_val_score(clf, X_sub, y_data, cv=3, scoring='accuracy', n_jobs=-1)
    return scores.mean()

# Track best solution
best_fitness = -np.inf
best_subset_idx = list(range(n_features))
best_accuracy = 0.0

print(f"Configuration: {N_ANTS} ants x {N_ITERATIONS} iterations")
print(f"Feature pool: {n_features} features")
print(f"Lambda penalty: {LAMBDA_PENALTY}")
print()

for iteration in range(N_ITERATIONS):
    iter_best_fitness = -np.inf
    iter_best_subset = None
    iter_best_acc = 0.0

    for ant in range(N_ANTS):
        # Probabilistic feature selection based on pheromone + heuristic
        prob = (pheromone ** ALPHA) * (heuristic ** BETA)
        prob = prob / prob.sum()

        # Each ant selects a random number of features (at least MIN_FEATURES)
        n_select = np.random.randint(MIN_FEATURES, max(MIN_FEATURES+1, n_features))
        selected_idx = np.random.choice(n_features, size=min(n_select, n_features),
                                         replace=False, p=prob)
        selected_idx = sorted(selected_idx)

        # Evaluate fitness
        accuracy = evaluate_subset(selected_idx, X_ant, y_ant)
        n_sel = len(selected_idx)
        fitness = accuracy - LAMBDA_PENALTY * (n_sel / n_features)

        if fitness > iter_best_fitness:
            iter_best_fitness = fitness
            iter_best_subset = selected_idx
            iter_best_acc = accuracy

    # Update pheromone
    pheromone *= (1 - EVAPORATION)  # evaporation
    if iter_best_subset is not None:
        pheromone[iter_best_subset] += iter_best_fitness  # deposit

    # Clamp pheromone
    pheromone = np.clip(pheromone, 0.1, 10.0)

    # Update global best
    if iter_best_fitness > best_fitness:
        best_fitness = iter_best_fitness
        best_subset_idx = iter_best_subset
        best_accuracy = iter_best_acc

    print(f"  Iter {iteration+1:>2}/{N_ITERATIONS}  |  "
          f"Best fitness: {iter_best_fitness:.4f}  |  "
          f"Accuracy: {iter_best_acc:.4f}  |  "
          f"Features: {len(iter_best_subset) if iter_best_subset is not None else 0}")

# Extract best feature names
ant_selected_features = [ant_features[i] for i in best_subset_idx]
ant_selected_set = set(ant_selected_features)

print(f"\n{'='*55}")
print(f"Red Ant Best Solution:")
print(f"  Fitness:        {best_fitness:.4f}")
print(f"  Accuracy:       {best_accuracy:.4f}")
print(f"  Features:       {len(ant_selected_features)}")
print(f"  Selected:       {ant_selected_features}")
print(f"Completed in {time.time()-t0:.1f}s")

Stage 4: Red Ant Feature Selection Algorithm
Configuration: 20 ants x 15 iterations
Feature pool: 28 features
Lambda penalty: 0.05

  Iter  1/15  |  Best fitness: 0.9909  |  Accuracy: 0.9998  |  Features: 5
  Iter  2/15  |  Best fitness: 0.9890  |  Accuracy: 0.9998  |  Features: 6
  Iter  3/15  |  Best fitness: 0.9873  |  Accuracy: 0.9998  |  Features: 7
  Iter  4/15  |  Best fitness: 0.9906  |  Accuracy: 0.9996  |  Features: 5
  Iter  5/15  |  Best fitness: 0.9890  |  Accuracy: 0.9997  |  Features: 6
  Iter  6/15  |  Best fitness: 0.9906  |  Accuracy: 0.9995  |  Features: 5
  Iter  7/15  |  Best fitness: 0.9891  |  Accuracy: 0.9998  |  Features: 6
  Iter  8/15  |  Best fitness: 0.9909  |  Accuracy: 0.9998  |  Features: 5
  Iter  9/15  |  Best fitness: 0.9903  |  Accuracy: 0.9992  |  Features: 5
  Iter 10/15  |  Best fitness: 0.9873  |  Accuracy: 0.9998  |  Features: 7
  Iter 11/15  |  Best fitness: 0.9888  |  Accuracy: 0.9995  |  Features: 6
  Iter 12/15  |  Best fitness: 0.9891  |  A

In [9]:
# ======================================
# FEATURE SELECTION — STAGE 5
# Feature Aggregation Strategy
# ======================================
# Combine outputs from all selection methods using a voting/scoring
# approach. Features earn points for being selected by each method.

print("Stage 5: Feature Aggregation Strategy")
print("=" * 55)

t0 = time.time()

all_features = list(X_full.columns)

# Score each feature: +1 point per method that selects it
# Method 1: RF Importance - top 75% of features by importance
rf_top_n = max(10, int(len(all_features) * 0.75))
rf_top_features = set(rf_importance_df.head(rf_top_n)['Feature'].tolist())

# Method 2: Correlation Filter survivors
corr_survivors = set(features_after_corr)

# Method 3: RFE-XGBoost selected
rfe_set = rfe_selected_set

# Method 4: Red Ant selected
ant_set = ant_selected_set

# Build aggregation table
agg_records = []
for feat in all_features:
    score = 0
    methods = []
    if feat in rf_top_features:
        score += 1
        methods.append("RF")
    if feat in corr_survivors:
        score += 1
        methods.append("Corr")
    if feat in rfe_set:
        score += 1
        methods.append("RFE")
    if feat in ant_set:
        score += 1
        methods.append("Ant")
    agg_records.append({
        'Feature': feat,
        'Score': score,
        'RF_Importance': rf_importance_map.get(feat, 0),
        'Methods': ', '.join(methods)
    })

agg_df = pd.DataFrame(agg_records).sort_values(
    ['Score', 'RF_Importance'], ascending=[False, False]
).reset_index(drop=True)

# Select features with score >= 3 (selected by at least 3 out of 4 methods)
# If too few, lower threshold to 2
MIN_FINAL_FEATURES = 10
score_threshold = 3
final_features = agg_df[agg_df['Score'] >= score_threshold]['Feature'].tolist()

if len(final_features) < MIN_FINAL_FEATURES:
    score_threshold = 2
    final_features = agg_df[agg_df['Score'] >= score_threshold]['Feature'].tolist()

print("\nAggregation Table (all features):")
print(f"{'Feature':<20}{'Score':<7}{'RF_Importance':<15}{'Methods'}")
print("-" * 65)
for _, row in agg_df.iterrows():
    marker = " [SELECTED]" if row['Feature'] in final_features else ""
    print(f"{row['Feature']:<20}{row['Score']:<7}{row['RF_Importance']:<15.6f}{row['Methods']}{marker}")

print(f"\nSelection threshold: score >= {score_threshold}")
print(f"Final selected features: {len(final_features)}")
print(f"Features: {final_features}")
print(f"\nCompleted in {time.time()-t0:.1f}s")

Stage 5: Feature Aggregation Strategy

Aggregation Table (all features):
Feature             Score  RF_Importance  Methods
-----------------------------------------------------------------
IdleTime            4      0.146801       RF, Corr, RFE, Ant [SELECTED]
sTtl                4      0.095358       RF, Corr, RFE, Ant [SELECTED]
pLoss               4      0.033417       RF, Corr, RFE, Ant [SELECTED]
DAppBytes           3      0.056016       RF, Corr, Ant [SELECTED]
TcpRtt              3      0.054974       RF, Corr, RFE [SELECTED]
DIntPkt             3      0.054439       RF, Corr, RFE [SELECTED]
SrcLoss             3      0.047522       RF, Corr, RFE [SELECTED]
DstRate             3      0.027381       RF, Corr, RFE [SELECTED]
SrcBytes            3      0.020607       RF, Corr, RFE [SELECTED]
DstPkts             3      0.018057       RF, Corr, RFE [SELECTED]
dTtl                3      0.016837       RF, Corr, RFE [SELECTED]
RunTime             3      0.016744       RF, Corr, Ant [SE

In [10]:
# ======================================
# APPLY FINAL FEATURE SUBSET & PREPROCESSING
# ======================================
# Filter the dataset to only the selected features.
# Apply scaling/normalization for downstream models.

print("Applying Feature Selection Results")
print("=" * 55)

# Apply final feature subset
X = X_full[final_features].copy()
y = y_full.copy()

print(f"Feature matrix after selection: {X.shape}")
print(f"Features: {list(X.columns)}")

# Handle any remaining NaN / Inf (safety)
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)

# StandardScaler for normalization
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print(f"\nPreprocessing complete:")
print(f"  - NaN/Inf handled")
print(f"  - StandardScaler applied")
print(f"  - Final shape: {X_scaled.shape}")

Applying Feature Selection Results
Feature matrix after selection: (1194464, 16)
Features: ['IdleTime', 'sTtl', 'pLoss', 'DAppBytes', 'TcpRtt', 'DIntPkt', 'SrcLoss', 'DstRate', 'SrcBytes', 'DstPkts', 'dTtl', 'RunTime', 'TotBytes', 'SIntPkt', 'SrcPkts', 'DstLoss']

Preprocessing complete:
  - NaN/Inf handled
  - StandardScaler applied
  - Final shape: (1194464, 16)


In [11]:
# ======================================
# TRAIN-TEST SPLIT
# ======================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples:  {len(X_test):,}")
print(f"Feature count:    {X_train.shape[1]}")

Training samples: 955,571
Testing samples:  238,893
Feature count:    16


In [12]:
# ======================================
# SMOTE OVERSAMPLING (TRAINING DATA ONLY)
# ======================================
# Applies SMOTE to balance class distribution in training data.
# Test data remains untouched to preserve real-world evaluation.

print("Class distribution BEFORE SMOTE:")
print(y_train.value_counts().sort_index())
print()

t0 = time.time()

smote = SMOTE(random_state=RANDOM_SEED)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("Class distribution AFTER SMOTE:")
print(y_train.value_counts().sort_index())
print(f"\nTraining samples after SMOTE: {len(X_train):,}")
print(f"SMOTE completed in {time.time()-t0:.1f}s")

Class distribution BEFORE SMOTE:
Traffic
0       170
1       207
2     62644
3      6592
4    885958
Name: count, dtype: int64

Class distribution AFTER SMOTE:
Traffic
0    885958
1    885958
2    885958
3    885958
4    885958
Name: count, dtype: int64

Training samples after SMOTE: 4,429,790
SMOTE completed in 9.0s


In [ ]:
# ======================================
# MODEL TRAINING (GPU OPTIMIZED)
# ======================================
# Train XGBoost, LightGBM, and Random Forest on the optimized feature set.

num_classes = y_train.nunique()
print(f"Number of classes: {num_classes}")
print(f"Training features: {X_train.shape[1]}")

t_start = time.time()

# -------------------------------
# XGBoost (GPU)
# -------------------------------
print("\nTraining XGBoost (GPU)...")
t0 = time.time()
xgb = XGBClassifier(
    n_estimators=300,
    objective="multi:softmax",
    num_class=num_classes,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    random_state=RANDOM_SEED,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    verbosity=0
)
xgb.fit(X_train, y_train)
print(f"  XGBoost done in {time.time()-t0:.1f}s")

# -------------------------------
# LightGBM (CPU)
# -------------------------------
print("\nTraining LightGBM...")
t0 = time.time()
lgbm = LGBMClassifier(
    n_estimators=300,
    objective="multiclass",
    num_class=num_classes,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    verbose=-1,
    force_row_wise=True
)
lgbm.fit(X_train, y_train)
print(f"  LightGBM done in {time.time()-t0:.1f}s")

# -------------------------------
# Random Forest
# -------------------------------
print("\nTraining Random Forest...")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    class_weight='balanced'
)
rf.fit(X_train, y_train)
print(f"  Random Forest done in {time.time()-t0:.1f}s")

print(f"\nAll models trained in {time.time()-t_start:.1f}s")

Number of classes: 5
Training features: 16

Training XGBoost (GPU)...
  XGBoost done in 32.3s

Training LightGBM...


In [ ]:
# ======================================
# SHAP EXPLAINABILITY
# ======================================
# Compute SHAP values for the XGBoost model to verify
# that selected features contribute meaningfully.

import shap

print("Running SHAP analysis (this may take 1-2 minutes)...")
t0 = time.time()

# Background and explanation samples
background = X_train.sample(200, random_state=RANDOM_SEED)
sample_data = X_train.sample(300, random_state=RANDOM_SEED)

explainer = shap.Explainer(xgb.predict, background)
shap_values = explainer(sample_data)

# Compute mean |SHAP| per feature
feature_importance = np.mean(np.abs(shap_values.values), axis=0)
shap_df = pd.DataFrame({
    'Feature': X_train.columns,
    'SHAP_Importance': feature_importance
}).sort_values('SHAP_Importance', ascending=False).reset_index(drop=True)

print(f"\nSHAP analysis completed in {time.time()-t0:.1f}s")
print(f"\nSHAP Feature Importance (selected features):")
print(f"{'Feature':<25}{'Mean |SHAP|':<15}")
print("-" * 40)
for _, row in shap_df.iterrows():
    print(f"{row['Feature']:<25}{row['SHAP_Importance']:.6f}")

# Verify all selected features have non-trivial SHAP importance
zero_shap = shap_df[shap_df['SHAP_Importance'] < 1e-6]
if len(zero_shap) == 0:
    print("\nAll selected features have meaningful SHAP contributions.")
else:
    print(f"\nWARNING: {len(zero_shap)} features have near-zero SHAP importance:")
    print(zero_shap['Feature'].tolist())

# Compute feature weights (normalized SHAP importance)
normalized_shap = feature_importance / feature_importance.max()
feature_weights = 0.5 + 0.5 * normalized_shap  # scale to [0.5, 1.0]
print(f"\nFeature weights computed (range: {feature_weights.min():.3f} - {feature_weights.max():.3f})")

In [ ]:
# ======================================
# FEATURE WEIGHTING
# ======================================
# Scale feature values using SHAP importance to amplify
# influential signals before ensemble prediction.

weights_vector = np.array(feature_weights)

X_train_w = X_train * weights_vector
X_test_w = X_test * weights_vector

print("Dynamic SHAP-based feature weighting applied.")
print(f"Weight range: [{weights_vector.min():.4f}, {weights_vector.max():.4f}]")

In [ ]:
# ======================================
# ENSEMBLE MODEL (SOFT VOTING)
# ======================================
# Combine predictions using soft voting (probability averaging).

from sklearn.ensemble import VotingClassifier

print("Training Hybrid Soft Voting Ensemble (GPU)...")
t0 = time.time()

hybrid_model = VotingClassifier(
    estimators=[
        ('xgb', XGBClassifier(
            n_estimators=300,
            objective="multi:softmax",
            num_class=num_classes,
            eval_metric="mlogloss",
            tree_method="hist",
            device="cuda",
            random_state=RANDOM_SEED,
            max_depth=10,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            verbosity=0
        )),
        ('lgbm', LGBMClassifier(
            n_estimators=300,
            objective="multiclass",
            num_class=num_classes,
            max_depth=10,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_SEED,
            verbose=-1,
            force_row_wise=True
        )),
        ('rf', RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            n_jobs=-1,
            random_state=RANDOM_SEED,
            class_weight='balanced'
        ))
    ],
    voting='soft',
    n_jobs=-1
)

hybrid_model.fit(X_train_w, y_train)
preds = hybrid_model.predict(X_test_w)

print(f"Ensemble training completed in {time.time()-t0:.1f}s")

In [ ]:
# ======================================
# EVALUATION METRICS
# ======================================

accuracy = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds, average='weighted')
recall = recall_score(y_test, preds, average='weighted')
f1 = f1_score(y_test, preds, average='weighted')

print("\n========== FINAL PERFORMANCE ==========")
print(f"Accuracy : {accuracy:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall   : {recall:.6f}")
print(f"F1-Score : {f1:.6f}")

print("\n========== CLASSIFICATION REPORT ==========")
class_names = le.classes_
print(classification_report(y_test, preds, target_names=class_names, digits=4))

print("========== CONFUSION MATRIX ==========")
cm = confusion_matrix(y_test, preds)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print(cm_df)

# Summary
print("\n========== PIPELINE SUMMARY ==========")
print(f"Original features:    {len(X_full.columns)}")
print(f"After correlation:    {len(features_after_corr)}")
print(f"RFE selected:         {len(rfe_selected)}")
print(f"Red Ant selected:     {len(ant_selected_features)}")
print(f"Final (aggregated):   {len(final_features)}")
print(f"Models in ensemble:   XGBoost (GPU), LightGBM, Random Forest")
print(f"SHAP weighted:        Yes")
print(f"SMOTE applied:        Yes (training only)")

# --- Extended Evaluation & Validation ---
> All cells below are **read-only evaluation blocks** that do NOT modify the trained model.

In [ ]:
# ======================================
# EVALUATION 1 - CLASS DISTRIBUTION VISUALIZATION
# ======================================
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

class_names = le.classes_
class_counts = y.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0'][:len(class_names)]
bars = ax.bar(class_names, class_counts.values, color=colors, edgecolor='white', linewidth=1.2)

for bar, count in zip(bars, class_counts.values):
    pct = count / class_counts.sum() * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Class Distribution (Full Dataset)', fontsize=14, fontweight='bold')
ax.set_xlabel('Traffic Class')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: class_distribution.png")

In [ ]:
# ======================================
# EVALUATION 2 - DETAILED PERFORMANCE METRICS TABLE
# ======================================
from sklearn.metrics import precision_score, recall_score, f1_score

class_names_list = le.classes_

accuracy_val  = accuracy_score(y_test, preds)
macro_f1  = f1_score(y_test, preds, average='macro')
weighted_f1 = f1_score(y_test, preds, average='weighted')
macro_prec = precision_score(y_test, preds, average='macro')
weighted_prec = precision_score(y_test, preds, average='weighted')
macro_rec = recall_score(y_test, preds, average='macro')
weighted_rec = recall_score(y_test, preds, average='weighted')

print("=" * 55)
print("OVERALL METRICS")
print("=" * 55)
print(f"Accuracy:          {accuracy_val:.4f}")
print(f"Macro Precision:   {macro_prec:.4f}")
print(f"Macro Recall:      {macro_rec:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted Precision:{weighted_prec:.4f}")
print(f"Weighted Recall:   {weighted_rec:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")

print("\n" + "=" * 55)
print("PER-CLASS REPORT")
print("=" * 55)
print(classification_report(y_test, preds, target_names=class_names_list, digits=4))

In [ ]:
# ======================================
# EVALUATION 3 - CONFUSION MATRIX VISUALIZATION
# ======================================
import matplotlib.pyplot as plt

class_names_list = le.classes_
cm = confusion_matrix(y_test, preds)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: Count Matrix
im0 = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[0].text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=9)

# Right: Percentage Matrix
im1 = axes[1].imshow(cm_pct, interpolation='nearest', cmap='Oranges')
axes[1].set_title('Confusion Matrix (%)', fontsize=13, fontweight='bold')
for i in range(cm_pct.shape[0]):
    for j in range(cm_pct.shape[1]):
        axes[1].text(j, i, f'{cm_pct[i,j]:.1f}%', ha='center', va='center',
                     color='white' if cm_pct[i,j] > 50 else 'black', fontsize=9)

for ax in axes:
    ax.set_xticks(range(len(class_names_list)))
    ax.set_yticks(range(len(class_names_list)))
    ax.set_xticklabels(class_names_list, rotation=45, ha='right')
    ax.set_yticklabels(class_names_list)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
# ======================================
# EVALUATION 4 - 5-FOLD CROSS VALIDATION
# ======================================
# Uses fresh model clones. Does NOT touch the main trained model.

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

print("Running 5-Fold Stratified Cross-Validation...")
print("(Trains fresh model copies per fold - does NOT touch the main model)")
print()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_w, y_train), 1):
    t0 = time.time()

    X_f_train, X_f_val = X_train_w.iloc[train_idx], X_train_w.iloc[val_idx]
    y_f_train, y_f_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    fold_model = clone(hybrid_model)
    fold_model.fit(X_f_train, y_f_train)
    fold_preds = fold_model.predict(X_f_val)

    fold_acc = accuracy_score(y_f_val, fold_preds)
    fold_f1 = f1_score(y_f_val, fold_preds, average='weighted')

    fold_results.append({'Fold': fold, 'Accuracy': fold_acc, 'Weighted_F1': fold_f1})
    print(f"  Fold {fold}: Accuracy={fold_acc:.4f}  F1={fold_f1:.4f}  ({time.time()-t0:.1f}s)")

cv_df = pd.DataFrame(fold_results)
print(f"\nMean Accuracy: {cv_df['Accuracy'].mean():.4f} +/- {cv_df['Accuracy'].std():.4f}")
print(f"Mean F1:       {cv_df['Weighted_F1'].mean():.4f} +/- {cv_df['Weighted_F1'].std():.4f}")

In [ ]:
# ======================================
# EVALUATION 5 - BASELINE MODEL COMPARISON
# ======================================
# Trains lightweight baselines on the SAME weighted features for fair comparison.

from sklearn.linear_model import LogisticRegression

print("Training Baseline Models for Comparison...")
print("(Using same SHAP-weighted features)")
print()

baselines = {}

# Logistic Regression
t0 = time.time()
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, n_jobs=-1)
lr.fit(X_train_w, y_train)
lr_preds = lr.predict(X_test_w)
baselines['Logistic Regression'] = {
    'Accuracy': accuracy_score(y_test, lr_preds),
    'Macro F1': f1_score(y_test, lr_preds, average='macro'),
    'Weighted F1': f1_score(y_test, lr_preds, average='weighted')
}
print(f"  Logistic Regression: {time.time()-t0:.1f}s")

# Standalone Random Forest
t0 = time.time()
rf_base = RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=RANDOM_SEED)
rf_base.fit(X_train_w, y_train)
rf_preds = rf_base.predict(X_test_w)
baselines['Random Forest'] = {
    'Accuracy': accuracy_score(y_test, rf_preds),
    'Macro F1': f1_score(y_test, rf_preds, average='macro'),
    'Weighted F1': f1_score(y_test, rf_preds, average='weighted')
}
print(f"  Random Forest: {time.time()-t0:.1f}s")

# Standalone XGBoost
t0 = time.time()
xgb_base = XGBClassifier(n_estimators=100, tree_method='hist', device='cuda',
                          random_state=RANDOM_SEED, verbosity=0)
xgb_base.fit(X_train_w, y_train)
xgb_preds = xgb_base.predict(X_test_w)
baselines['XGBoost'] = {
    'Accuracy': accuracy_score(y_test, xgb_preds),
    'Macro F1': f1_score(y_test, xgb_preds, average='macro'),
    'Weighted F1': f1_score(y_test, xgb_preds, average='weighted')
}
print(f"  XGBoost: {time.time()-t0:.1f}s")

# Our Ensemble
baselines['Hybrid Ensemble'] = {
    'Accuracy': accuracy_score(y_test, preds),
    'Macro F1': f1_score(y_test, preds, average='macro'),
    'Weighted F1': f1_score(y_test, preds, average='weighted')
}

# Print comparison table
print(f"\n{'Model':<25}{'Accuracy':<12}{'Macro F1':<12}{'Weighted F1':<12}")
print("-" * 60)
for model, metrics in baselines.items():
    print(f"{model:<25}{metrics['Accuracy']:<12.4f}{metrics['Macro F1']:<12.4f}{metrics['Weighted F1']:<12.4f}")

In [ ]:
# ======================================
# EVALUATION 6 - COMPARISON BAR CHART
# ======================================
import matplotlib.pyplot as plt

model_names = list(baselines.keys())
metrics_keys = ['Accuracy', 'Macro F1', 'Weighted F1']
colors = ['#607D8B', '#9E9E9E', '#78909C', '#1976D2']

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(metrics_keys))
width = 0.18
offsets = np.linspace(-(len(model_names)-1)*width/2,
                       (len(model_names)-1)*width/2, len(model_names))

for idx, (model, offset) in enumerate(zip(model_names, offsets)):
    values = [baselines[model][m] for m in metrics_keys]
    bars = ax.bar(x + offset, values, width, label=model, color=colors[idx % len(colors)])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_keys, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Comparison - Feature-Selected Pipeline', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")